In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from prophet import Prophet
from helper import apply_hte, add_deterministic_features
import os

RANDOM_STATE = 42
BASE = os.path.dirname(os.getcwd())
DIR = os.path.join(BASE, "data-round-1")

In [2]:
best_xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.07589566183731224,
    'max_depth': 4,
    'objective': 'reg:pseudohubererror',
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'subsample': 0.8332913820261092,
    'colsample_bytree': 0.9956754283056524,
    'min_child_weight': 6
}

best_lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.018445672558531936,
    'max_depth': 5,
    'objective': 'huber',
    'random_state': RANDOM_STATE,
    'verbose': -1,
    'num_leaves': 44,
    'subsample': 0.664271846321548,
    'colsample_bytree': 0.8053629230501607
}

best_cat_params = {
    'iterations': 2000,
    'learning_rate': 0.022760622703184193,
    'depth': 8,
    'loss_function': 'Huber:delta=1.5',
    'random_state': RANDOM_STATE,
    'verbose': False,
    'l2_leaf_reg': 9.170733265281925
}


In [9]:
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import GridSearchCV

# ---------------------------------------------------------
# 3. STACKING PIPELINE WITH HUBER LOSS
# ---------------------------------------------------------
def run_stacking_pipeline(train_path, test_path, best_xgb_params = best_xgb_params, best_lgb_params = best_lgb_params, best_cat_params = best_cat_params):
    print("1. Data Prep & Deterministic Features...")
    df = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    df['Revenue_log'] = np.log1p(df['Revenue'])
    df['COGS_Ratio'] = df['COGS'] / (df['Revenue'] + 1e-9) 
    
    df = add_deterministic_features(df)
    df_test = add_deterministic_features(df_test)
    
    static_features = [
        'year', 'month', 'day', 'dayofweek',
        'days_to_next_tet', 'tet_pre_effect', 'tet_post_effect',
        'is_sale_season', 'sale_rank', 'days_to_next_sale' 
        # 'sale_progress', , 'days_since_last_sale', 'sale_pre_effect', 'sale_post_effect'
    ]
    
    folds = [
        ('2012-07-04', '2018-12-31', '2019-01-01', '2019-12-31'),
        ('2012-07-04', '2019-12-31', '2020-01-01', '2020-12-31'),
        ('2012-07-04', '2020-12-31', '2021-01-01', '2021-12-31'),
        ('2012-07-04', '2021-12-31', '2022-01-01', '2022-12-31')
    ]
    
    # Cấu hình Base Models (Tất cả dùng Huber Loss)
    xgb_params = {'n_estimators': 2000, 'learning_rate': 0.03, 'max_depth': 6, 'objective': 'reg:pseudohubererror',  'tree_method': 'hist', 'random_state': RANDOM_STATE}
    lgb_params = {'n_estimators': 2000, 'learning_rate': 0.03, 'max_depth': 6, 'objective': 'huber',  'random_state': RANDOM_STATE, 'verbose': -1}
    cat_params = {'iterations': 2000, 'learning_rate': 0.03, 'depth': 6, 'loss_function': 'Huber:delta=1.5',  'random_state': RANDOM_STATE, 'verbose': False}
    
    xgb_params.update(best_xgb_params)
    lgb_params.update(best_lgb_params)
    cat_params.update(best_cat_params)

    print(f"best_xgb_params = {xgb_params}")
    print(f"best_lgb_params = {lgb_params}")
    print(f"best_cat_params = {cat_params}")
    
    # Arrays thu thập dự đoán Out-Of-Fold (OOF) để huấn luyện Trọng tài
    oof_rev_xgb, oof_rev_lgb, oof_rev_cat = [], [], []
    oof_rat_xgb, oof_rat_lgb, oof_rat_cat = [], [], []
    oof_y_rev, oof_y_rat = [], []
    
    print("2. Walk-Forward CV & Generating OOF Predictions...")
    for i, (train_start, train_end, val_start, val_end) in enumerate(folds):
        train_fold = df[(df['Date'] >= train_start) & (df['Date'] <= train_end)].copy()
        val_fold = df[(df['Date'] >= val_start) & (df['Date'] <= val_end)].copy()
        
        train_fold['hte_Revenue_log'] = apply_hte(train_fold, train_fold, ['month', 'dayofweek'], 'Revenue_log')
        val_fold['hte_Revenue_log']   = apply_hte(train_fold, val_fold, ['month', 'dayofweek'], 'Revenue_log')
        train_fold['hte_COGS_Ratio']  = apply_hte(train_fold, train_fold, ['month', 'dayofweek'], 'COGS_Ratio')
        val_fold['hte_COGS_Ratio']    = apply_hte(train_fold, val_fold, ['month', 'dayofweek'], 'COGS_Ratio')
        
        # Trend Phóng
        prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        prophet_df = train_fold[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
        prophet.fit(prophet_df)
        
        train_trend = prophet.predict(prophet_df)['yhat'].values
        val_trend = prophet.predict(val_fold[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
        train_fold['residual'] = train_fold['Revenue_log'] - train_trend
        val_fold['residual'] = val_fold['Revenue_log'] - val_trend
        
        features = static_features + ['hte_Revenue_log', 'hte_COGS_Ratio']
        X_tr, X_va = train_fold[features], val_fold[features]
        
        # --- TRAIN BASE MODELS (REVENUE) ---
        m_rev_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
        m_rev_xgb.fit(X_tr, train_fold['residual'], eval_set=[(X_va, val_fold['residual'])], verbose=False)
        
        m_rev_lgb = lgb.LGBMRegressor(**lgb_params)
        m_rev_lgb.fit(X_tr, train_fold['residual'], eval_set=[(X_va, val_fold['residual'])], callbacks=[lgb.early_stopping(50, verbose=False)])
        
        m_rev_cat = CatBoostRegressor(**cat_params, early_stopping_rounds=50)
        m_rev_cat.fit(X_tr, train_fold['residual'], eval_set=[(X_va, val_fold['residual'])], verbose=False)
        
        # Thu thập log OOF Revenue 
        log_p_xgb_rev = val_trend + m_rev_xgb.predict(X_va)
        log_p_lgb_rev = val_trend + m_rev_lgb.predict(X_va)
        log_p_cat_rev = val_trend + m_rev_cat.predict(X_va)
        
        oof_rev_xgb.extend(log_p_xgb_rev)
        oof_rev_lgb.extend(log_p_lgb_rev)
        oof_rev_cat.extend(log_p_cat_rev)
        oof_y_rev.extend(val_fold['Revenue_log'])
        
        # --- TRAIN BASE MODELS (COGS RATIO) ---
        m_rat_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
        m_rat_xgb.fit(X_tr, train_fold['COGS_Ratio'], eval_set=[(X_va, val_fold['COGS_Ratio'])], verbose=False)
        
        m_rat_lgb = lgb.LGBMRegressor(**lgb_params)
        m_rat_lgb.fit(X_tr, train_fold['COGS_Ratio'], eval_set=[(X_va, val_fold['COGS_Ratio'])], callbacks=[lgb.early_stopping(50, verbose=False)])
        
        m_rat_cat = CatBoostRegressor(**cat_params, early_stopping_rounds=50)
        m_rat_cat.fit(X_tr, train_fold['COGS_Ratio'], eval_set=[(X_va, val_fold['COGS_Ratio'])], verbose=False)
        
        # Thu thập OOF Ratio
        oof_rat_xgb.extend(m_rat_xgb.predict(X_va))
        oof_rat_lgb.extend(m_rat_lgb.predict(X_va))
        oof_rat_cat.extend(m_rat_cat.predict(X_va))
        oof_y_rat.extend(val_fold['COGS_Ratio'])

    # ---------------------------------------------------------
    # 4. TRAIN META-MODELS (TRỌNG TÀI HUBER)
    # ---------------------------------------------------------
    print("3. Training Meta-Models (HuberRegressor)...")
    X_meta_rev = np.column_stack([oof_rev_xgb, oof_rev_lgb, oof_rev_cat])
    # Revenue Meta Tuning
    huber_grid = {'epsilon': [1.1, 1.2, 1.35, 1.5], 'alpha': [0.0001, 0.001, 0.01, 0.1]}
    meta_rev_search = GridSearchCV(HuberRegressor(fit_intercept=False), huber_grid, cv=5, scoring='neg_mean_absolute_error')
    meta_rev_search.fit(X_meta_rev, oof_y_rev)
    meta_rev_model = meta_rev_search.best_estimator_
    
    X_meta_rat = np.column_stack([oof_rat_xgb, oof_rat_lgb, oof_rat_cat])
    # Ratio Meta Tuning (Epsilon nhỏ hơn vì khoảng giá trị nhỏ)
    huber_ratio_grid = {'epsilon': [1.01, 1.05, 1.1, 1.2], 'alpha': [0.0001, 0.001, 0.01, 0.1]}
    meta_rat_search = GridSearchCV(HuberRegressor(fit_intercept=False), huber_ratio_grid, cv=5, scoring='neg_mean_absolute_error')
    meta_rat_search.fit(X_meta_rat, oof_y_rat)
    meta_rat_model = meta_rat_search.best_estimator_
    
    print(f" -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Revenue: {meta_rev_model.coef_}")
    print(f" -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Ratio: {meta_rat_model.coef_}")

    # ---------------------------------------------------------
    # 5. FINAL TRAIN ON FULL DATA & INFERENCE
    # ---------------------------------------------------------
    print("4. Training Full Base Models for Test Set...")
    df['hte_Revenue_log'] = apply_hte(df, df, ['month', 'dayofweek'], 'Revenue_log')
    df_test['hte_Revenue_log'] = apply_hte(df, df_test, ['month', 'dayofweek'], 'Revenue_log')
    df['hte_COGS_Ratio'] = apply_hte(df, df, ['month', 'dayofweek'], 'COGS_Ratio')
    df_test['hte_COGS_Ratio'] = apply_hte(df, df_test, ['month', 'dayofweek'], 'COGS_Ratio')
    
    final_features = static_features + ['hte_Revenue_log', 'hte_COGS_Ratio']
    
    # Bỏ early_stopping
    xgb_params.pop('early_stopping_rounds', None)
    cat_params.pop('early_stopping_rounds', None)
    xgb_params['n_estimators'] = 700
    lgb_params['n_estimators'] = 700
    cat_params['iterations'] = 700
    
    final_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    prophet_df_full = df[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
    final_prophet.fit(prophet_df_full)
    df['residual'] = df['Revenue_log'] - final_prophet.predict(prophet_df_full)['yhat'].values
    
    # Train Revenue Full
    f_rev_xgb, f_rev_lgb, f_rev_cat = xgb.XGBRegressor(**xgb_params), lgb.LGBMRegressor(**lgb_params), CatBoostRegressor(**cat_params)
    f_rev_xgb.fit(df[final_features], df['residual'], verbose=False)
    f_rev_lgb.fit(df[final_features], df['residual'])
    f_rev_cat.fit(df[final_features], df['residual'], verbose=False)
    
    # Train Ratio Full
    f_rat_xgb, f_rat_lgb, f_rat_cat = xgb.XGBRegressor(**xgb_params), lgb.LGBMRegressor(**lgb_params), CatBoostRegressor(**cat_params)
    f_rat_xgb.fit(df[final_features], df['COGS_Ratio'], verbose=False)
    f_rat_lgb.fit(df[final_features], df['COGS_Ratio'])
    f_rat_cat.fit(df[final_features], df['COGS_Ratio'], verbose=False)
    
    print("5. Generating Meta-Predictions...")
    test_trend = final_prophet.predict(df_test[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
    
    # Base Predictions (Revenue)
    test_log_p_rev_xgb = test_trend + f_rev_xgb.predict(df_test[final_features])
    test_log_p_rev_lgb = test_trend + f_rev_lgb.predict(df_test[final_features])
    test_log_p_rev_cat = test_trend + f_rev_cat.predict(df_test[final_features])
    
    # Base Predictions (Ratio)
    test_p_rat_xgb = f_rat_xgb.predict(df_test[final_features])
    test_p_rat_lgb = f_rat_lgb.predict(df_test[final_features])
    test_p_rat_cat = f_rat_cat.predict(df_test[final_features])
    
    # Meta Model Inference
    X_test_meta_rev = np.column_stack([test_log_p_rev_xgb, test_log_p_rev_lgb, test_log_p_rev_cat])
    X_test_meta_rat = np.column_stack([test_p_rat_xgb, test_p_rat_lgb, test_p_rat_cat])
    
    final_log_rev = meta_rev_model.predict(X_test_meta_rev)
    df_test['Revenue'] = np.expm1(final_log_rev)
    df_test['COGS_Ratio_Pred'] = meta_rat_model.predict(X_test_meta_rat)
    
    df_test['COGS'] = df_test['Revenue'] * df_test['COGS_Ratio_Pred']
    df_test['Revenue'] = df_test['Revenue'].clip(lower=0)
    df_test['COGS'] = df_test['COGS'].clip(lower=0)
    
    target_folder = os.path.join(BASE, "Submission")
    os.makedirs(target_folder, exist_ok= True)
    submission_name = os.path.join(target_folder, "submission.csv")
    submission = df_test[['Date', 'Revenue', 'COGS']]
    submission.to_csv(submission_name, index=False)
    print(f"Done! Saved to {submission_name}")

In [10]:
train_path = os.path.join(DIR, "sales.csv")
test_path = os.path.join(DIR, "sample_submission.csv")

In [11]:
run_stacking_pipeline(train_path, test_path)

1. Data Prep & Deterministic Features...
best_xgb_params = {'n_estimators': 2000, 'learning_rate': 0.07589566183731224, 'max_depth': 4, 'objective': 'reg:pseudohubererror', 'tree_method': 'hist', 'random_state': 42, 'subsample': 0.8332913820261092, 'colsample_bytree': 0.9956754283056524, 'min_child_weight': 6}
best_lgb_params = {'n_estimators': 2000, 'learning_rate': 0.018445672558531936, 'max_depth': 5, 'objective': 'huber', 'random_state': 42, 'verbose': -1, 'num_leaves': 44, 'subsample': 0.664271846321548, 'colsample_bytree': 0.8053629230501607}
best_cat_params = {'iterations': 2000, 'learning_rate': 0.022760622703184193, 'depth': 8, 'loss_function': 'Huber:delta=1.5', 'random_state': 42, 'verbose': False, 'l2_leaf_reg': 9.170733265281925}
2. Walk-Forward CV & Generating OOF Predictions...


09:17:39 - cmdstanpy - INFO - Chain [1] start processing
09:17:40 - cmdstanpy - INFO - Chain [1] done processing
09:17:45 - cmdstanpy - INFO - Chain [1] start processing
09:17:45 - cmdstanpy - INFO - Chain [1] done processing
09:17:51 - cmdstanpy - INFO - Chain [1] start processing
09:17:52 - cmdstanpy - INFO - Chain [1] done processing
09:17:59 - cmdstanpy - INFO - Chain [1] start processing
09:18:00 - cmdstanpy - INFO - Chain [1] done processing


3. Training Meta-Models (HuberRegressor)...
 -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Revenue: [0.44994064 0.49207148 0.06369242]
 -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Ratio: [-0.72569193  0.68349814  1.12077331]
4. Training Full Base Models for Test Set...


09:18:16 - cmdstanpy - INFO - Chain [1] start processing
09:18:17 - cmdstanpy - INFO - Chain [1] done processing


5. Generating Meta-Predictions...
Done! Saved to d:\FPT\SU26\DATATHON26\Submission\submission.csv
